In [1]:
%pip install langchain-text-splitters langchain-community langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 7.7 MB/s eta 0:00:007.6 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 7.6 MB/s eta 0:00:00 MB/s eta 0:00:01
  Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl (14.0 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.0/456.0 kB 7.6 MB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.7/325.7 kB 7.4 MB/s eta 0:00:008.7 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.8 MB/s eta 0:00:008.2 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%pip install -qU langchain-mistralai


[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -qU langchain-ollama


[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install -U jq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.3/422.3 kB 6.9 MB/s eta 0:00:00 MB/s eta 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install -U faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.9 MB/s eta 0:00:006.3 MB/s eta 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Mistal chat about John Constable/Louvre - https://chat.mistral.ai/chat/c0092bfc-823e-4de4-a8b7-abf6a230e48c

In [4]:
import os

os.environ["MISTRAL_API_KEY"] = "bssa46y3DAugMuEf0GArQpZEhNp8c4fp"

from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(model="mistral-large-latest")

In [259]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(base_url="http://192.168.178.130:11434", model="all-minilm:l6-v2")

In [260]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [19]:
from langchain import hub
from langchain_community.document_loaders import CSVLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import json


In [279]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='/Users/digitalmediaadmin/Git/etc-docs/notebooks/ai-explorations/datasets/vam/dogs-101/good-dogs-object-data.csv',
    source_column='SystemNumber',
    content_columns=['Description', 'Title', 'SystemNumber'],
    csv_args={
    'delimiter': ',',
    'quotechar': '"'
})

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

In [285]:
len(all_splits)

121

In [284]:

# Index chunks
_ = vector_store.add_documents(documents=all_splits)


In [286]:
vector_store.index.ntotal

350

In [263]:
vector_store.index.reconstruct_n()[2]

array([ 3.77890058e-02,  4.99146804e-02,  9.82362032e-02, -5.58294076e-03,
        1.35842366e-02,  2.44043544e-02, -1.79504771e-02, -3.75769543e-03,
       -5.87500865e-03,  1.98780727e-02, -5.64914793e-02,  1.59478150e-02,
       -3.46522294e-02,  1.61775425e-02, -5.77652082e-02,  3.93343307e-02,
       -5.10313101e-02, -6.96608098e-03,  4.68765907e-02, -3.82176414e-03,
       -4.33531329e-02,  4.33286792e-03,  8.50157347e-03, -3.03194020e-02,
       -2.45160386e-02,  1.00120306e-02,  2.51961928e-02,  1.89993177e-02,
        5.25744110e-02,  2.77949590e-02, -2.80149337e-02, -4.84209098e-02,
       -3.47221382e-02,  4.37378548e-02,  3.41604054e-02,  7.22365454e-02,
       -7.27877300e-03, -9.07506852e-04, -4.22298396e-03,  9.38116387e-02,
        3.56254801e-02,  3.68400849e-02, -4.60310839e-02, -4.66129519e-02,
       -4.21934463e-02, -3.31955915e-03, -1.41136155e-01, -2.44638603e-02,
       -4.59427275e-02, -2.76061427e-02,  1.73303783e-02, -4.03607413e-02,
       -7.13758171e-03, -

In [281]:
results = vector_store.similarity_search(query="what artworks were owned by the Gilberts ?")
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}] ]")

* They could even be made to resemble full-sized canvas paintings, and indeed Arthur Gilbert himself mistook his very first micromosaic for a painting. When he brought it home to show his wife, he had to convince her that it was not in fact a cracked painting, as she supposed, but a mosaic.Sir Arthur Gilbert and his wife Rosalinde formed one of the world's great decorative art collections, including silver, mosaics, enamelled portrait miniatures and gold boxes. Arthur Gilbert donated his extraordinary collection to Britain in 1996. [{'source': 'O156393', 'row': 73}] ]
* sizes. They could even be made to resemble full-sized canvas paintings, and indeed Arthur Gilbert himself mistook his very first micromosaic for a painting. When he brought it home to show his wife, he had to convince her that it was not in fact a cracked painting, as she supposed, but a mosaic.Sir Arthur Gilbert and his wife Rosalinde formed one of the world's great decorative art collections, including silver, mosaics

In [59]:
# Retrieve museum object specific chat prompt to make responses more relevant for artworks

prompt = hub.pull("vam/chat-glam", api_url = "https://eu.api.smith.langchain.com/")

/Users/digitalmediaadmin/.pyenv/versions/3.9.18/envs/etc-docs/lib/python3.9/site-packages/langsmith/client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [222]:

# https://eu.smith.langchain.com/hub/vam/chat-glam
# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = db2.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

# Test Questions 

In [273]:
response = graph.invoke({"question": "What was the name of Hogarth's dog?"})
print(response["answer"])

The name of Hogarth's dog was Trump (SystemNumber: O97313). This pug was prominently featured in Hogarth's self-portrait of 1745. Hogarth's combative personality led him to be depicted as Trump in a 1753 caricature by Paul Sandby.


In [268]:
response = graph.invoke({"question": "What artworks are associated with greyhounds"})
print(response["answer"])

Two artworks are associated with greyhounds. The first is a painting by Louis Godefroy Jadin (no system number provided), which features greyhounds in a hunting scene. The second is a terracotta statuette of a hound (inv. no. A.10-1954), although it's not clear if it's specifically a greyhound.


In [275]:
response = graph.invoke({"question": "Which artworks depict St Dominic with a dog?"})
print(response["answer"])

The artwork that depicts St. Dominic with a dog is a stone and walnut ivory relief made by Diego Reinoso in 1669. The dog, which resembles a Chinese Foo dog, is sitting at the feet of St. Dominic and is one of his main symbols.


In [276]:
response = graph.invoke({"question": "How many artworks have greyhounds in them?"})
print(response["answer"])

Based on the provided context, there is only one artwork that mentions greyhounds. The artwork is described in the context with system number O123500.


In [277]:
response = graph.invoke({"question": "How many artworks have dogs in them?"})
print(response["answer"])

Based on the provided context, there are 3 artworks that have dogs in them. Two are studies of a dog (no system number provided) and one is a terracotta statuette of a hound (System Number: O72976).


In [289]:
response = graph.invoke({"question": "How many artworks have Spaniels in them?"})
print(response["answer"])

I don't know. There is no mention of Spaniels in the provided context. The artworks described include a Mastiff (O72971) and terracotta statuettes of hounds (A.10-1954 and A.11-1954).


In [288]:
response = graph.invoke({"question": "How many artworks have newfoundlands in them?"})
print(response["answer"])

Based on the provided context, there is one artwork that features a Newfoundland dog. The artwork is listed with the system number O69349, titled "Lion: A Newfoundland Dog".


In [215]:
response = graph.invoke({"question": "How many artworks are there?"})
print(response["answer"])

I don't know the total number of artworks, but based on the provided context, there are at least 4 artworks mentioned.


In [291]:
response = graph.invoke({"question": "Which artworks mention Cruft's?"})
print(response["answer"])

I don't know. None of the provided artworks mention "Cruft's."
